In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
import yaml

# ---- Paths ----
DRIVE_ROOT = "/content/drive/MyDrive/pathology_project"   # persistent, large files
REPO_ROOT  = "/content/pathology_project"                  # this becomes your git repo

# Drive-side folders (large/persistent stuff)
drive_folders = ["checkpoints", "embeddings", "data"]
for f in drive_folders:
    os.makedirs(os.path.join(DRIVE_ROOT, f), exist_ok=True)

# Repo-side folders (git-tracked stuff)
repo_folders = ["configs", "src", "scripts", "splits", "results", "figures", "logs"]
for f in repo_folders:
    os.makedirs(os.path.join(REPO_ROOT, f), exist_ok=True)

# ---- Config file (lives in repo, tracked by git) ----
config = {
    "project_name": "breast_cancer_pathology_prototype",
    "seed": 42,
    "paths": {
        "drive_root": DRIVE_ROOT,
        "repo_root": REPO_ROOT,
        "checkpoints": os.path.join(DRIVE_ROOT, "checkpoints"),
        "embeddings": os.path.join(DRIVE_ROOT, "embeddings"),
        "data": os.path.join(DRIVE_ROOT, "data"),
        "splits": os.path.join(REPO_ROOT, "splits"),
        "results": os.path.join(REPO_ROOT, "results"),
        "logs": os.path.join(REPO_ROOT, "logs"),
    },
    "split_ratios": {"train": 0.70, "val": 0.15, "test": 0.15},
    "status": {"day": 1, "step_completed": "step0_project_init"},
}

config_path = os.path.join(REPO_ROOT, "configs", "config.yaml")
with open(config_path, "w") as f:
    yaml.dump(config, f, default_flow_style=False)

# ---- .gitignore so large files never accidentally get pushed ----
gitignore_content = """\
# Large / binary artifacts — kept in Drive, not GitHub
checkpoints/
embeddings/
data/
*.pt
*.pth
*.npy
*.npz
__pycache__/
*.pyc
.ipynb_checkpoints/
"""
with open(os.path.join(REPO_ROOT, ".gitignore"), "w") as f:
    f.write(gitignore_content)

print("✅ Drive mounted and persistent folders created at:", DRIVE_ROOT)
print("✅ Repo (git-ready) structure created at:", REPO_ROOT)
print("✅ Config saved at:", config_path)
print("✅ .gitignore created")

In [ ]:
!cat /content/pathology_project/configs/config.yaml

In [ ]:
from google.colab import files
print("Upload your kaggle.json file:")
uploaded = files.upload()   # kaggle.json select karo

In [ ]:
import os

os.makedirs("/root/.kaggle", exist_ok=True)
os.system("cp kaggle.json /root/.kaggle/kaggle.json")
os.system("chmod 600 /root/.kaggle/kaggle.json")

!pip install -q kagglehub
import kagglehub

# BreakHis dataset (most common Kaggle version)
path = kagglehub.dataset_download("ambarish/breakhis")

print("✅ Dataset downloaded at:", path)

In [ ]:
import os
import re
import pandas as pd

BASE = "/root/.cache/kagglehub/datasets/ambarish/breakhis/versions/4"

# ---- Full directory walk (top 4 levels) ----
print("📁 Directory structure (sample):\n")
count = 0
for dirpath, dirnames, filenames in os.walk(BASE):
    depth = dirpath[len(BASE):].count(os.sep)
    if depth <= 3:
        print("  " * depth + os.path.basename(dirpath) + f"  ({len(filenames)} files)")
    if depth > 4:
        dirnames[:] = []
    count += 1
    if count > 200:
        print("... (truncated)")
        break

# ---- Collect all image files ----
image_records = []
for dirpath, dirnames, filenames in os.walk(BASE):
    for fname in filenames:
        if fname.lower().endswith((".png", ".jpg", ".jpeg")):
            image_records.append({
                "filepath": os.path.join(dirpath, fname),
                "filename": fname,
                "dirpath": dirpath,
            })

df = pd.DataFrame(image_records)
print(f"\n📊 Total images found: {len(df)}")

# ---- Show sample filenames + paths ----
print("\n🔎 Sample filenames:")
print(df["filename"].sample(min(10, len(df))).to_list())

print("\n🔎 Sample full paths:")
for p in df["filepath"].sample(min(5, len(df))).to_list():
    print(" -", p)

# ---- Try to detect patient ID pattern from BreakHis filename convention ----
# BreakHis filenames typically look like: SOB_B_A-14-22549AB-40-001.png
# Pattern: SOB_<class>_<subtype>-<patient_id>-<mag>-<seq>.png
pattern = re.compile(r"SOB_([BM])_([A-Z]+)-(\d+-\d+)-(\d+)-(\d+)")

matches = df["filename"].apply(lambda f: pattern.match(f))
matched_count = matches.notna().sum()
print(f"\n🧬 Filenames matching expected BreakHis pattern: {matched_count} / {len(df)}")

if matched_count > 0:
    sample_match = matches[matches.notna()].iloc[0]
    print("Example parsed groups:", sample_match.groups())

In [ ]:
import os
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit

BASE = "/root/.cache/kagglehub/datasets/ambarish/breakhis/versions/4"

# ---- Rebuild image list with group ID from folder structure ----
records = []
for dirpath, dirnames, filenames in os.walk(BASE):
    for fname in filenames:
        if fname.lower().endswith((".png", ".jpg", ".jpeg")):
            full_path = os.path.join(dirpath, fname)
            parts = full_path.split(os.sep)
            # .../breast/<benign|malignant>/SOB/<tumor_type>/<patient_folder>/<mag>/<file>
            try:
                mag_folder = parts[-2]              # e.g. "400X"
                patient_folder = parts[-3]           # e.g. "SOB_M_DC_14-17915"
                tumor_type = parts[-4]               # e.g. "ductal_carcinoma"
                label = parts[-6]                    # "benign" or "malignant"
            except IndexError:
                continue
            records.append({
                "filepath": full_path,
                "filename": fname,
                "group_id": patient_folder,
                "magnification": mag_folder,
                "tumor_type": tumor_type,
                "label": label,
            })

df = pd.DataFrame(records)
print(f"📊 Total images: {len(df)}")
print(f"🧬 Unique patient/group folders: {df['group_id'].nunique()}")

# ---- STEP 2: Verify group identifier quality ----
print("\n🔎 Sample group IDs:")
print(df["group_id"].sample(5).to_list())

imgs_per_group = df.groupby("group_id").size()
print(f"\n📈 Images per group -> min: {imgs_per_group.min()}, max: {imgs_per_group.max()}, mean: {imgs_per_group.mean():.1f}")

label_counts = df["label"].value_counts()
print(f"\n🏷️ Label counts:\n{label_counts}")

# Sanity check: does every group map to exactly one label?
group_label_check = df.groupby("group_id")["label"].nunique()
bad_groups = group_label_check[group_label_check > 1]
print(f"\n⚠️ Groups with inconsistent labels: {len(bad_groups)} (should be 0)")

# ---- STEP 3: Patient-wise 70/15/15 split ----
SEED = 42
groups = df["group_id"].unique()

gss1 = GroupShuffleSplit(n_splits=1, train_size=0.70, random_state=SEED)
train_idx, temp_idx = next(gss1.split(df, groups=df["group_id"]))

train_df = df.iloc[train_idx]
temp_df = df.iloc[temp_idx]

gss2 = GroupShuffleSplit(n_splits=1, train_size=0.50, random_state=SEED)  # 50% of remaining 30% = 15/15
val_idx, test_idx = next(gss2.split(temp_df, groups=temp_df["group_id"]))

val_df = temp_df.iloc[val_idx]
test_df = temp_df.iloc[test_idx]

print(f"\n✅ Split sizes -> Train: {len(train_df)} imgs / {train_df['group_id'].nunique()} groups")
print(f"✅ Split sizes -> Val:   {len(val_df)} imgs / {val_df['group_id'].nunique()} groups")
print(f"✅ Split sizes -> Test:  {len(test_df)} imgs / {test_df['group_id'].nunique()} groups")

# ---- STEP 4: Leakage check ----
train_groups = set(train_df["group_id"])
val_groups = set(val_df["group_id"])
test_groups = set(test_df["group_id"])

overlap_train_val = train_groups & val_groups
overlap_train_test = train_groups & test_groups
overlap_val_test = val_groups & test_groups

dup_paths = pd.concat([train_df["filepath"], val_df["filepath"], test_df["filepath"]])
dup_check = dup_paths.duplicated().sum()

leakage_pass = (
    len(overlap_train_val) == 0 and
    len(overlap_train_test) == 0 and
    len(overlap_val_test) == 0 and
    dup_check == 0
)

print(f"\n🧪 Leakage check: {'✅ PASS' if leakage_pass else '❌ FAIL'}")
print(f"   train∩val: {len(overlap_train_val)}, train∩test: {len(overlap_train_test)}, val∩test: {len(overlap_val_test)}")
print(f"   duplicate paths across splits: {dup_check}")

# ---- Save splits + leakage report ----
REPO_ROOT = "/content/pathology_project"
os.makedirs(os.path.join(REPO_ROOT, "splits"), exist_ok=True)
os.makedirs(os.path.join(REPO_ROOT, "results"), exist_ok=True)

train_df.to_csv(f"{REPO_ROOT}/splits/train.csv", index=False)
val_df.to_csv(f"{REPO_ROOT}/splits/val.csv", index=False)
test_df.to_csv(f"{REPO_ROOT}/splits/test.csv", index=False)

with open(f"{REPO_ROOT}/results/leakage_report.txt", "w") as f:
    f.write(f"Leakage check: {'PASS' if leakage_pass else 'FAIL'}\n")
    f.write(f"train groups: {len(train_groups)}, val groups: {len(val_groups)}, test groups: {len(test_groups)}\n")
    f.write(f"train∩val: {len(overlap_train_val)}, train∩test: {len(overlap_train_test)}, val∩test: {len(overlap_val_test)}\n")
    f.write(f"duplicate paths: {dup_check}\n")
    f.write(f"seed: {SEED}\n")

print("\n💾 Saved: splits/train.csv, splits/val.csv, splits/test.csv, results/leakage_report.txt")

In [ ]:
import torch
import time
from PIL import Image
import torchvision.transforms as T

print(f"🖥️ GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"   Device: {torch.cuda.get_device_name(0)}")

# ---- Load DINOv2 ViT-S/14 ----
t0 = time.time()
model = torch.hub.load('facebookresearch/dinov2', 'dinov2_vits14')
model.eval()
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
print(f"✅ Model loaded in {time.time()-t0:.1f}s on {device}")

# ---- Load one real BreakHis image ----
sample_path = train_df.iloc[0]["filepath"]
img = Image.open(sample_path).convert("RGB")
print(f"✅ Loaded sample image: {sample_path}, size={img.size}")

# ---- Preprocess (DINOv2 expects 224x224, ImageNet norm, patch-14 -> multiple of 14) ----
transform = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])
img_tensor = transform(img).unsqueeze(0).to(device)

# ---- Run inference ----
t0 = time.time()
with torch.no_grad():
    embedding = model(img_tensor)
infer_time = time.time() - t0

print(f"✅ Embedding generated in {infer_time:.3f}s")
print(f"✅ Embedding shape: {embedding.shape}")

if torch.cuda.is_available():
    mem_mb = torch.cuda.memory_allocated() / 1024**2
    print(f"✅ GPU memory used: {mem_mb:.1f} MB")

# ---- Save feasibility report ----
report = f"""Foundation Model Feasibility Report
Model: DINOv2 ViT-S/14
Device: {device}
Model load time: {time.time()-t0:.1f}s
Sample image: {sample_path}
Embedding shape: {tuple(embedding.shape)}
Inference time: {infer_time:.3f}s
GPU memory (MB): {mem_mb if torch.cuda.is_available() else 'N/A'}
Status: PASS
"""
with open(f"{REPO_ROOT}/results/foundation_feasibility_report.txt", "w") as f:
    f.write(report)

print("\n💾 Saved: results/foundation_feasibility_report.txt")

In [ ]:
configs/config.yaml
splits/train.csv
splits/val.csv
splits/test.csv
results/leakage_report.txt
results/foundation_feasibility_report.txt
.gitignore


In [ ]:
# %cd /content/pathology_project

# !git init
# !git config user.email "abdulrehman2024131@example.com"
# !git config user.name "abdul-rehman-24"

# !git add configs/ splits/ results/ .gitignore
# !git commit -m "Day 1: leakage-safe dataset split and foundation feasibility"

In [ ]:
# !git remote add origin https://github.com/abdul-rehman-24/breast-histopathology-patch-classification.git
# !git branch -M main
# !git push -u origin main

In [ ]:
# !git push -u origin main

In [ ]:
import yaml

config_path = "/content/pathology_project/configs/config.yaml"
with open(config_path) as f:
    config = yaml.safe_load(f)

config["paths"]["data"] = "/root/.cache/kagglehub/datasets/ambarish/breakhis/versions/4"
config["paths"]["data_note"] = "Raw dataset NOT stored in Drive — re-downloaded fresh each session via kagglehub (fast, ~30s, ~4GB)"

with open(config_path, "w") as f:
    yaml.dump(config, f, default_flow_style=False)

print("✅ Config updated with correct data path note")

In [ ]:
# %cd /content/pathology_project
# !git add configs/config.yaml
# !git commit -m "Day 1: clarify raw data storage strategy (kagglehub cache, not Drive)"
# !git push

In [ ]:
# ---- RUN AGAIN: environment setup ----
from google.colab import drive
drive.mount('/content/drive')

import os, yaml, pandas as pd, torch

# ---- Clone your GitHub repo (contains Day-1 splits/config) ----
REPO_ROOT = "/content/pathology_project"
if not os.path.exists(REPO_ROOT):
    !git clone https://github.com/abdul-rehman-24/breast-histopathology-patch-classification.git {REPO_ROOT}
else:
    %cd {REPO_ROOT}
    !git pull

# ---- LOAD saved Day-1 artifacts ----
config_path = f"{REPO_ROOT}/configs/config.yaml"
with open(config_path) as f:
    config = yaml.safe_load(f)

train_df = pd.read_csv(f"{REPO_ROOT}/splits/train.csv")
val_df   = pd.read_csv(f"{REPO_ROOT}/splits/val.csv")
test_df  = pd.read_csv(f"{REPO_ROOT}/splits/test.csv")

print("✅ Config loaded:", config["project_name"])
print(f"✅ Splits loaded -> train={len(train_df)}, val={len(val_df)}, test={len(test_df)}")

# ---- RUN AGAIN: re-download raw dataset (fast, not stored in Drive) ----
!pip install -q kagglehub
import kagglehub
DATA_PATH = kagglehub.dataset_download("ambarish/breakhis")
print("✅ Dataset re-downloaded at:", DATA_PATH)

# ---- IMPORTANT: filepaths in CSV may point to OLD session's cache path ----
# Kaggle sometimes reuses the same cache path, but let's verify:
sample_path = train_df.iloc[0]["filepath"]
print("\n🔎 Sample path from CSV:", sample_path)
print("🔎 Exists?:", os.path.exists(sample_path))
print("🔎 Current DATA_PATH:", DATA_PATH)

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import torchvision.transforms as T

# ---- Label encoding ----
LABEL_MAP = {"benign": 0, "malignant": 1}

class BreakHisDataset(Dataset):
    def __init__(self, df, data_root_fix=None, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform
        self.data_root_fix = data_root_fix  # (old_root, new_root) tuple if remap needed

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        path = row["filepath"]
        if self.data_root_fix:
            old_root, new_root = self.data_root_fix
            path = path.replace(old_root, new_root)
        img = Image.open(path).convert("RGB")
        if self.transform:
            img = self.transform(img)
        label = LABEL_MAP[row["label"]]
        return img, label

# ---- Transforms ----
train_transform = T.Compose([
    T.Resize((224, 224)),
    T.RandomHorizontalFlip(),
    T.RandomVerticalFlip(),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

eval_transform = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# ---- Determine if path remap needed (based on Step 0 check) ----
DATA_ROOT_FIX = None
sample_path = train_df.iloc[0]["filepath"]
if not os.path.exists(sample_path):
    # extract old root prefix up to 'BreaKHis_v1' and remap to new DATA_PATH
    old_root = sample_path.split("BreaKHis_v1")[0].rstrip("/")
    new_root = DATA_PATH
    DATA_ROOT_FIX = (old_root, new_root)
    print(f"⚠️ Remapping paths: {old_root} -> {new_root}")
else:
    print("✅ Paths still valid, no remap needed")

# ---- Build datasets + loaders ----
train_ds = BreakHisDataset(train_df, data_root_fix=DATA_ROOT_FIX, transform=train_transform)
val_ds   = BreakHisDataset(val_df,   data_root_fix=DATA_ROOT_FIX, transform=eval_transform)
test_ds  = BreakHisDataset(test_df,  data_root_fix=DATA_ROOT_FIX, transform=eval_transform)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True, num_workers=2)
val_loader   = DataLoader(val_ds, batch_size=32, shuffle=False, num_workers=2)
test_loader  = DataLoader(test_ds, batch_size=32, shuffle=False, num_workers=2)

# ---- Sanity check: one batch ----
imgs, labels = next(iter(train_loader))
print(f"✅ Batch image tensor shape: {imgs.shape}")
print(f"✅ Batch label tensor shape: {labels.shape}")
print(f"✅ Sample labels: {labels[:10].tolist()}")
print(f"✅ Image value range: [{imgs.min():.2f}, {imgs.max():.2f}]")

In [ ]:
import torchvision.models as models
import torch.nn as nn

device = "cuda" if torch.cuda.is_available() else "cpu"

def build_efficientnet_b0(num_classes=2, pretrained=True):
    weights = models.EfficientNet_B0_Weights.IMAGENET1K_V1 if pretrained else None
    model = models.efficientnet_b0(weights=weights)
    in_features = model.classifier[1].in_features
    model.classifier[1] = nn.Linear(in_features, num_classes)
    return model

model = build_efficientnet_b0(num_classes=2, pretrained=True).to(device)

# ---- Forward pass sanity check ----
model.eval()
with torch.no_grad():
    imgs_gpu = imgs.to(device)
    out = model(imgs_gpu)

print(f"✅ Model loaded on {device}")
print(f"✅ Output shape: {out.shape}")  # expect (32, 2)
print(f"✅ Sample logits: {out[0].cpu().numpy()}")
print(f"✅ Total trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

In [ ]:
import torch.optim as optim
from sklearn.metrics import roc_auc_score
import time, json

# ---- Training config ----
SEED = 42
torch.manual_seed(SEED)

EPOCHS = 10
LR = 1e-4
CHECKPOINT_DIR = "/content/drive/MyDrive/pathology_project/checkpoints/cnn_baseline"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# class weights (inverse frequency)
class_counts = train_df["label"].value_counts()
weight_benign = len(train_df) / (2 * class_counts["benign"])
weight_malignant = len(train_df) / (2 * class_counts["malignant"])
class_weights = torch.tensor([weight_benign, weight_malignant], dtype=torch.float32).to(device)

criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = optim.AdamW(model.parameters(), lr=LR)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=2, factor=0.5)

train_config = {
    "seed": SEED, "epochs": EPOCHS, "lr": LR, "batch_size": 32,
    "optimizer": "AdamW", "loss": "CrossEntropyLoss (weighted)",
    "scheduler": "ReduceLROnPlateau", "class_weights": class_weights.cpu().tolist(),
    "selection_metric": "val_auroc",
}
with open(f"{REPO_ROOT}/configs/cnn_training_config.json", "w") as f:
    json.dump(train_config, f, indent=2)
print("✅ Training config saved")

# ---- Resume support: check for existing checkpoint ----
start_epoch = 0
best_val_auroc = 0.0
latest_ckpt_path = f"{CHECKPOINT_DIR}/latest.pt"

if os.path.exists(latest_ckpt_path):
    ckpt = torch.load(latest_ckpt_path, map_location=device)
    model.load_state_dict(ckpt["model_state"])
    optimizer.load_state_dict(ckpt["optimizer_state"])
    scheduler.load_state_dict(ckpt["scheduler_state"])
    start_epoch = ckpt["epoch"] + 1
    best_val_auroc = ckpt["best_val_auroc"]
    print(f"🔄 Resuming from epoch {start_epoch}, best_val_auroc so far: {best_val_auroc:.4f}")
else:
    print("🆕 No checkpoint found, starting fresh")

# ---- Training loop ----
def run_epoch(loader, train_mode=True):
    model.train() if train_mode else model.eval()
    total_loss = 0
    all_labels, all_probs = [], []
    with torch.set_grad_enabled(train_mode):
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device)
            if train_mode:
                optimizer.zero_grad()
            out = model(imgs)
            loss = criterion(out, labels)
            if train_mode:
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * imgs.size(0)
            probs = torch.softmax(out, dim=1)[:, 1]
            all_labels.extend(labels.cpu().numpy())
            all_probs.extend(probs.detach().cpu().numpy())
    avg_loss = total_loss / len(loader.dataset)
    auroc = roc_auc_score(all_labels, all_probs)
    return avg_loss, auroc

for epoch in range(start_epoch, EPOCHS):
    t0 = time.time()
    train_loss, train_auroc = run_epoch(train_loader, train_mode=True)
    val_loss, val_auroc = run_epoch(val_loader, train_mode=False)
    scheduler.step(val_loss)

    print(f"Epoch {epoch+1}/{EPOCHS} | train_loss={train_loss:.4f} train_auroc={train_auroc:.4f} "
          f"| val_loss={val_loss:.4f} val_auroc={val_auroc:.4f} | {time.time()-t0:.1f}s")

    # save latest checkpoint (always, for resume)
    torch.save({
        "epoch": epoch, "model_state": model.state_dict(),
        "optimizer_state": optimizer.state_dict(), "scheduler_state": scheduler.state_dict(),
        "val_auroc": val_auroc, "best_val_auroc": max(best_val_auroc, val_auroc),
    }, latest_ckpt_path)

    # save best checkpoint (only if improved)
    if val_auroc > best_val_auroc:
        best_val_auroc = val_auroc
        torch.save({
            "epoch": epoch, "model_state": model.state_dict(),
            "val_auroc": val_auroc,
        }, f"{CHECKPOINT_DIR}/best.pt")
        print(f"  💾 New best checkpoint saved (val_auroc={val_auroc:.4f})")

print(f"\n✅ Training complete. Best val_auroc: {best_val_auroc:.4f}")

In [ ]:
best_ckpt = torch.load(f"{CHECKPOINT_DIR}/best.pt", map_location=device, weights_only=False)

In [ ]:
import numpy as np
from sklearn.metrics import (roc_auc_score, average_precision_score, f1_score,
                               precision_score, recall_score, accuracy_score,
                               balanced_accuracy_score, confusion_matrix)
import matplotlib.pyplot as plt

# ---- Load best checkpoint ----
best_ckpt = torch.load(f"{CHECKPOINT_DIR}/best.pt", map_location=device, weights_only=False)
model.load_state_dict(best_ckpt["model_state"])
model.eval()
print(f"✅ Loaded best checkpoint from epoch {best_ckpt['epoch']+1}, val_auroc={best_ckpt['val_auroc']:.4f}")

# ---- Run inference on TEST set (once) ----
all_labels, all_probs, all_preds = [], [], []
with torch.no_grad():
    for imgs, labels in test_loader:
        imgs = imgs.to(device)
        out = model(imgs)
        probs = torch.softmax(out, dim=1)[:, 1]
        preds = (probs > 0.5).long()
        all_labels.extend(labels.numpy())
        all_probs.extend(probs.cpu().numpy())
        all_preds.extend(preds.cpu().numpy())

all_labels = np.array(all_labels)
all_probs = np.array(all_probs)
all_preds = np.array(all_preds)

# ---- Metrics ----
cm = confusion_matrix(all_labels, all_preds)
tn, fp, fn, tp = cm.ravel()
sensitivity = tp / (tp + fn)
specificity = tn / (tn + fp)

metrics = {
    "auroc": roc_auc_score(all_labels, all_probs),
    "auprc": average_precision_score(all_labels, all_probs),
    "f1": f1_score(all_labels, all_preds),
    "precision": precision_score(all_labels, all_preds),
    "sensitivity_recall": sensitivity,
    "specificity": specificity,
    "accuracy": accuracy_score(all_labels, all_preds),
    "balanced_accuracy": balanced_accuracy_score(all_labels, all_preds),
    "confusion_matrix": cm.tolist(),
    "n_test_samples": len(all_labels),
    "checkpoint_epoch": best_ckpt["epoch"] + 1,
    "checkpoint_val_auroc": float(best_ckpt["val_auroc"]),
}

print("\n📊 TEST SET METRICS (CNN Baseline - EfficientNet-B0):")
for k, v in metrics.items():
    if k != "confusion_matrix":
        print(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")

# ---- Save metrics ----
os.makedirs(f"{REPO_ROOT}/results", exist_ok=True)
with open(f"{REPO_ROOT}/results/cnn_baseline_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

# ---- Confusion matrix figure ----
fig, ax = plt.subplots(figsize=(5,4))
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks([0,1]); ax.set_yticks([0,1])
ax.set_xticklabels(["Benign","Malignant"]); ax.set_yticklabels(["Benign","Malignant"])
ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")
ax.set_title("CNN Baseline - Confusion Matrix (Test)")
for i in range(2):
    for j in range(2):
        ax.text(j, i, cm[i,j], ha="center", va="center", color="black")
plt.colorbar(im)
os.makedirs(f"{REPO_ROOT}/figures", exist_ok=True)
plt.savefig(f"{REPO_ROOT}/figures/cnn_baseline_confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()

print("\n💾 Saved: results/cnn_baseline_metrics.json, figures/cnn_baseline_confusion_matrix.png")

In [ ]:
experiment_record = {
    "experiment_name": "cnn_baseline_efficientnet_b0",
    "model": "EfficientNet-B0 (ImageNet pretrained)",
    "dataset": "BreakHis",
    "split_version": "day1_patient_wise_70_15_15_seed42",
    "seed": 42,
    "training_config": train_config,
    "checkpoint_used": "best.pt (epoch 1, selected via val_auroc)",
    "checkpoint_location": f"{CHECKPOINT_DIR}/best.pt (Google Drive)",
    "validation_results": {"val_auroc": 0.9501, "epoch": 1},
    "test_results": metrics,
    "notes": "Model overfit after epoch 1 (train_auroc -> ~1.0 while val_auroc declined). "
             "Best checkpoint selected strictly via validation AUROC, not test. "
             "Test evaluated once, no tuning performed post-hoc. "
             "This is the CNN baseline for comparison with later Foundation-model and fusion experiments."
}

with open(f"{REPO_ROOT}/results/cnn_baseline_experiment_record.json", "w") as f:
    json.dump(experiment_record, f, indent=2)

print("✅ Experiment record saved")

# ---- Commit to GitHub ----
%cd {REPO_ROOT}
!git add configs/cnn_training_config.json results/cnn_baseline_metrics.json results/cnn_baseline_experiment_record.json figures/cnn_baseline_confusion_matrix.png
!git commit -m "Day 2: EfficientNet-B0 CNN baseline - metrics, experiment record"
!git push

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, yaml, json, pandas as pd, torch

REPO_ROOT = "/content/pathology_project"
if not os.path.exists(REPO_ROOT):
    !git clone https://github.com/abdul-rehman-24/breast-histopathology-patch-classification.git {REPO_ROOT}
else:
    %cd {REPO_ROOT}
    !git pull

with open(f"{REPO_ROOT}/configs/config.yaml") as f:
    config = yaml.safe_load(f)

train_df = pd.read_csv(f"{REPO_ROOT}/splits/train.csv")
val_df   = pd.read_csv(f"{REPO_ROOT}/splits/val.csv")
test_df  = pd.read_csv(f"{REPO_ROOT}/splits/test.csv")
print(f"✅ Splits loaded -> train={len(train_df)}, val={len(val_df)}, test={len(test_df)}")

# ---- re-download raw dataset ----
!pip install -q kagglehub
import kagglehub
DATA_PATH = kagglehub.dataset_download("ambarish/breakhis")
print("✅ Dataset re-downloaded at:", DATA_PATH)

# ---- path remap check (same as Day 2) ----
sample_path = train_df.iloc[0]["filepath"]
DATA_ROOT_FIX = None
if not os.path.exists(sample_path):
    old_root = sample_path.split("BreaKHis_v1")[0].rstrip("/")
    DATA_ROOT_FIX = (old_root, DATA_PATH)
    print(f"⚠️ Remap needed: {old_root} -> {DATA_PATH}")
else:
    print("✅ Paths valid, no remap needed")

# ---- verify Day-2 CNN checkpoint exists (untouched, not reloaded now) ----
CNN_CKPT = "/content/drive/MyDrive/pathology_project/checkpoints/cnn_baseline/best.pt"
print(f"✅ Day-2 CNN checkpoint exists: {os.path.exists(CNN_CKPT)}")

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

FOUNDATION_MODEL_NAME = "dinov2_vits14"
EMBEDDING_DIM = 384

foundation_model = torch.hub.load('facebookresearch/dinov2', FOUNDATION_MODEL_NAME)
foundation_model.eval()
foundation_model.to(device)

# ---- Freeze all parameters ----
for param in foundation_model.parameters():
    param.requires_grad = False

n_trainable = sum(p.numel() for p in foundation_model.parameters() if p.requires_grad)
n_total = sum(p.numel() for p in foundation_model.parameters())

print(f"✅ Model: {FOUNDATION_MODEL_NAME}")
print(f"✅ Device: {device}")
print(f"✅ Total params: {n_total:,}")
print(f"✅ Trainable params: {n_trainable:,} (should be 0)")
print(f"✅ Model in eval mode: {not foundation_model.training}")

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

FOUNDATION_MODEL_NAME = "dinov2_vits14"
EMBEDDING_DIM = 384

foundation_model = torch.hub.load('facebookresearch/dinov2', FOUNDATION_MODEL_NAME)
foundation_model.eval()
foundation_model.to(device)

# ---- Freeze all parameters ----
for param in foundation_model.parameters():
    param.requires_grad = False

n_trainable = sum(p.numel() for p in foundation_model.parameters() if p.requires_grad)
n_total = sum(p.numel() for p in foundation_model.parameters())

print(f"✅ Model: {FOUNDATION_MODEL_NAME}")
print(f"✅ Device: {device}")
print(f"✅ Total params: {n_total:,}")
print(f"✅ Trainable params: {n_trainable:,} (should be 0)")
print(f"✅ Model in eval mode: {not foundation_model.training}")

In [ ]:
import torchvision.transforms as T
from PIL import Image

dinov2_transform = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

sample_path = train_df.iloc[0]["filepath"]
if DATA_ROOT_FIX:
    sample_path = sample_path.replace(DATA_ROOT_FIX[0], DATA_ROOT_FIX[1])

img = Image.open(sample_path).convert("RGB")
img_tensor = dinov2_transform(img).unsqueeze(0).to(device)

with torch.no_grad():
    embedding = foundation_model(img_tensor)

print(f"✅ Input tensor shape: {img_tensor.shape}")
print(f"✅ Embedding shape: {embedding.shape}")
print(f"✅ Embedding dtype: {embedding.dtype}")
print(f"✅ Embedding device: {embedding.device}")

In [ ]:
EMB_DIR = "/content/drive/MyDrive/pathology_project/embeddings/dinov2_vits14"
os.makedirs(EMB_DIR, exist_ok=True)

def extract_embeddings(df, split_name, n_samples=None, resume=True):
    """Extract embeddings with resume support. n_samples=None means full split."""
    cache_path = f"{EMB_DIR}/{split_name}_embeddings.pt"

    subset_df = df.head(n_samples) if n_samples else df

    # ---- Load existing cache if resuming ----
    if resume and os.path.exists(cache_path):
        cache = torch.load(cache_path, weights_only=False)
        done_paths = set(cache["filepath"])
        print(f"🔄 Found existing cache with {len(done_paths)} embeddings")
    else:
        cache = {"filepath": [], "group_id": [], "label": [], "embedding": []}
        done_paths = set()

    remaining = subset_df[~subset_df["filepath"].isin(done_paths)]
    print(f"📊 {split_name}: {len(done_paths)} cached, {len(remaining)} remaining to extract")

    for idx, row in remaining.iterrows():
        path = row["filepath"]
        real_path = path.replace(DATA_ROOT_FIX[0], DATA_ROOT_FIX[1]) if DATA_ROOT_FIX else path
        img = Image.open(real_path).convert("RGB")
        img_tensor = dinov2_transform(img).unsqueeze(0).to(device)
        with torch.no_grad():
            emb = foundation_model(img_tensor).cpu().squeeze(0)
        cache["filepath"].append(path)
        cache["group_id"].append(row["group_id"])
        cache["label"].append(row["label"])
        cache["embedding"].append(emb)

    torch.save(cache, cache_path)
    print(f"✅ Saved {len(cache['filepath'])} total embeddings -> {cache_path}")
    return cache

# ---- SMALL TEST: only 5 images per split ----
test_cache_train = extract_embeddings(train_df, "train_TEST", n_samples=5, resume=False)
test_cache_val   = extract_embeddings(val_df, "val_TEST", n_samples=5, resume=False)
test_cache_test  = extract_embeddings(test_df, "test_TEST", n_samples=5, resume=False)

# ---- Verify ----
print(f"\n🔎 Sample embedding shape: {test_cache_train['embedding'][0].shape}")
print(f"🔎 Sample label: {test_cache_train['label'][0]}, group: {test_cache_train['group_id'][0]}")

# ---- Reload test to verify save/load works ----
reloaded = torch.load(f"{EMB_DIR}/train_TEST_embeddings.pt", weights_only=False)
print(f"✅ Reload check: {len(reloaded['filepath'])} embeddings, matches original: {len(reloaded['filepath']) == 5}")

In [ ]:
# ---- cleanup test files ----
for f in ["train_TEST_embeddings.pt", "val_TEST_embeddings.pt", "test_TEST_embeddings.pt"]:
    p = f"{EMB_DIR}/{f}"
    if os.path.exists(p):
        os.remove(p)
print("🧹 Test cache files removed")

from torch.utils.data import Dataset, DataLoader

class ExtractionDataset(Dataset):
    """Loads raw images for embedding extraction (no labels needed at load time)."""
    def __init__(self, df, transform):
        self.df = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        path = row["filepath"]
        real_path = path.replace(DATA_ROOT_FIX[0], DATA_ROOT_FIX[1]) if DATA_ROOT_FIX else path
        img = Image.open(real_path).convert("RGB")
        img = self.transform(img)
        return img, path, row["group_id"], row["label"]

def extract_embeddings_full(df, split_name, batch_size=64, save_every_n_batches=10):
    cache_path = f"{EMB_DIR}/{split_name}_embeddings.pt"

    if os.path.exists(cache_path):
        cache = torch.load(cache_path, weights_only=False)
        done_paths = set(cache["filepath"])
    else:
        cache = {"filepath": [], "group_id": [], "label": [], "embedding": []}
        done_paths = set()

    remaining_df = df[~df["filepath"].isin(done_paths)].reset_index(drop=True)
    total = len(df)
    print(f"📊 {split_name}: {len(done_paths)}/{total} already cached, {len(remaining_df)} remaining")

    if len(remaining_df) == 0:
        print(f"✅ {split_name}: nothing to do, already complete")
        return cache

    ds = ExtractionDataset(remaining_df, dinov2_transform)
    loader = DataLoader(ds, batch_size=batch_size, shuffle=False, num_workers=2)

    batch_count = 0
    for imgs, paths, group_ids, labels in loader:
        imgs = imgs.to(device)
        with torch.no_grad():
            embs = foundation_model(imgs).cpu()

        cache["filepath"].extend(list(paths))
        cache["group_id"].extend(list(group_ids))
        cache["label"].extend(list(labels))
        cache["embedding"].extend(list(embs))

        batch_count += 1
        done_now = len(cache["filepath"])
        if batch_count % save_every_n_batches == 0:
            torch.save(cache, cache_path)
            print(f"  💾 checkpoint saved: {done_now}/{total} processed")

    # final save
    torch.save(cache, cache_path)
    print(f"✅ {split_name} COMPLETE: {len(cache['filepath'])}/{total} embeddings saved -> {cache_path}")
    return cache

# ---- Run for all three splits ----
train_cache = extract_embeddings_full(train_df, "train", batch_size=64)
val_cache   = extract_embeddings_full(val_df, "val", batch_size=64)
test_cache  = extract_embeddings_full(test_df, "test", batch_size=64)

In [ ]:
def validate_cache(df, cache, split_name):
    issues = []

    n_expected = len(df)
    n_cached = len(cache["filepath"])
    if n_expected != n_cached:
        issues.append(f"Count mismatch: expected {n_expected}, got {n_cached}")

    # duplicates
    dup_count = len(cache["filepath"]) - len(set(cache["filepath"]))
    if dup_count > 0:
        issues.append(f"{dup_count} duplicate filepaths in cache")

    # every split image has an embedding (set comparison)
    expected_paths = set(df["filepath"])
    cached_paths = set(cache["filepath"])
    missing = expected_paths - cached_paths
    extra = cached_paths - expected_paths
    if missing:
        issues.append(f"{len(missing)} images missing from cache")
    if extra:
        issues.append(f"{len(extra)} extra images in cache not in split")

    # embedding dimension consistency
    dims = set(e.shape[0] for e in cache["embedding"])
    if len(dims) != 1:
        issues.append(f"Inconsistent embedding dims: {dims}")

    # labels match split manifest
    cache_df = pd.DataFrame({"filepath": cache["filepath"], "cached_label": cache["label"]})
    merged = df.merge(cache_df, on="filepath", how="inner")
    label_mismatches = (merged["label"] != merged["cached_label"]).sum()
    if label_mismatches > 0:
        issues.append(f"{label_mismatches} label mismatches")

    # group_id matches
    cache_df2 = pd.DataFrame({"filepath": cache["filepath"], "cached_group": cache["group_id"]})
    merged2 = df.merge(cache_df2, on="filepath", how="inner")
    group_mismatches = (merged2["group_id"] != merged2["cached_group"]).sum()
    if group_mismatches > 0:
        issues.append(f"{group_mismatches} group_id mismatches")

    status = "PASS" if len(issues) == 0 else "FAIL"
    print(f"{'✅' if status=='PASS' else '❌'} {split_name}: {status}")
    for issue in issues:
        print(f"   ⚠️ {issue}")
    return status == "PASS", issues

train_ok, train_issues = validate_cache(train_df, train_cache, "train")
val_ok, val_issues     = validate_cache(val_df, val_cache, "val")
test_ok, test_issues   = validate_cache(test_df, test_cache, "test")

overall_pass = train_ok and val_ok and test_ok
print(f"\n🧪 OVERALL CACHE INTEGRITY: {'✅ PASS' if overall_pass else '❌ FAIL'}")

embedding_dim = train_cache["embedding"][0].shape[0]
print(f"📐 Embedding dimension: {embedding_dim}")

In [ ]:
class FoundationOnlyModel(nn.Module):
    def __init__(self, in_dim=384, proj_dim=256, num_classes=2):
        super().__init__()
        self.projection = ProjectionHead(in_dim, proj_dim)
        self.classifier = nn.Sequential(
            nn.Linear(proj_dim, 128), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(128, num_classes)
        )
    def forward(self, x):
        return self.classifier(self.projection(x))

foundation_only_model = FoundationOnlyModel().to(device)

class FoundationOnlyDataset(Dataset):
    def __init__(self, emb_cache):
        self.features = emb_cache["embedding"]
        self.labels = [LABEL_MAP[l] for l in emb_cache["label"]]
    def __len__(self): return len(self.labels)
    def __getitem__(self, idx): return self.features[idx], self.labels[idx]

train_fo_ds = FoundationOnlyDataset(train_emb_cache)
val_fo_ds   = FoundationOnlyDataset(val_emb_cache)
test_fo_ds  = FoundationOnlyDataset(test_emb_cache)
train_fo_loader = DataLoader(train_fo_ds, batch_size=64, shuffle=True)
val_fo_loader   = DataLoader(val_fo_ds, batch_size=64, shuffle=False)
test_fo_loader  = DataLoader(test_fo_ds, batch_size=64, shuffle=False)

FO_CKPT_DIR = "/content/drive/MyDrive/pathology_project/checkpoints/foundation_only"
os.makedirs(FO_CKPT_DIR, exist_ok=True)

torch.manual_seed(42)
fo_optimizer = optim.AdamW(foundation_only_model.parameters(), lr=1e-3)
fo_scheduler = optim.lr_scheduler.ReduceLROnPlateau(fo_optimizer, mode='min', patience=3, factor=0.5)
fo_criterion = nn.CrossEntropyLoss(weight=class_weights)

def run_fo_epoch(loader, model, train_mode=True):
    model.train() if train_mode else model.eval()
    total_loss = 0
    all_labels, all_probs = [], []
    with torch.set_grad_enabled(train_mode):
        for feats, labels in loader:
            feats, labels = feats.to(device), labels.to(device)
            if train_mode: fo_optimizer.zero_grad()
            out = model(feats)
            loss = fo_criterion(out, labels)
            if train_mode:
                loss.backward(); fo_optimizer.step()
            total_loss += loss.item() * feats.size(0)
            probs = torch.softmax(out, dim=1)[:, 1]
            all_labels.extend(labels.cpu().numpy())
            all_probs.extend(probs.detach().cpu().numpy())
    return total_loss/len(loader.dataset), roc_auc_score(all_labels, all_probs)

best_fo_val_auroc = 0.0
EPOCHS_FO = 20
for epoch in range(EPOCHS_FO):
    train_loss, train_auroc = run_fo_epoch(train_fo_loader, foundation_only_model, True)
    val_loss, val_auroc = run_fo_epoch(val_fo_loader, foundation_only_model, False)
    fo_scheduler.step(val_loss)
    print(f"Epoch {epoch+1}/{EPOCHS_FO} | train_auroc={train_auroc:.4f} | val_loss={val_loss:.4f} val_auroc={val_auroc:.4f}")
    if val_auroc > best_fo_val_auroc:
        best_fo_val_auroc = val_auroc
        torch.save({"epoch": epoch, "model_state": foundation_only_model.state_dict(), "val_auroc": float(val_auroc)},
                   f"{FO_CKPT_DIR}/best.pt")
        print(f"  💾 New best (val_auroc={val_auroc:.4f})")

print(f"\n✅ Foundation-only training complete. Best val_auroc: {best_fo_val_auroc:.4f}")

# ---- Test evaluation ----
best_fo_ckpt = torch.load(f"{FO_CKPT_DIR}/best.pt", map_location=device, weights_only=False)
foundation_only_model.load_state_dict(best_fo_ckpt["model_state"])
foundation_only_model.eval()

all_labels, all_probs, all_preds = [], [], []
with torch.no_grad():
    for feats, labels in test_fo_loader:
        feats = feats.to(device)
        out = foundation_only_model(feats)
        probs = torch.softmax(out, dim=1)[:, 1]
        preds = (probs > 0.5).long()
        all_labels.extend(labels.numpy())
        all_probs.extend(probs.cpu().numpy())
        all_preds.extend(preds.cpu().numpy())

all_labels, all_probs, all_preds = np.array(all_labels), np.array(all_probs), np.array(all_preds)
cm_fo = confusion_matrix(all_labels, all_preds)
tn, fp, fn, tp = cm_fo.ravel()

foundation_only_metrics = {
    "auroc": roc_auc_score(all_labels, all_probs),
    "auprc": average_precision_score(all_labels, all_probs),
    "f1": f1_score(all_labels, all_preds),
    "precision": precision_score(all_labels, all_preds),
    "sensitivity_recall": tp/(tp+fn),
    "specificity": tn/(tn+fp),
    "accuracy": accuracy_score(all_labels, all_preds),
    "balanced_accuracy": balanced_accuracy_score(all_labels, all_preds),
    "confusion_matrix": cm_fo.tolist(),
    "n_test_samples": len(all_labels),
    "checkpoint_epoch": best_fo_ckpt["epoch"]+1,
    "checkpoint_val_auroc": float(best_fo_ckpt["val_auroc"]),
}

print("\n📊 TEST METRICS (Foundation-Only Baseline):")
for k, v in foundation_only_metrics.items():
    if k != "confusion_matrix":
        print(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")

with open(f"{REPO_ROOT}/results/foundation_only_metrics.json", "w") as f:
    json.dump(foundation_only_metrics, f, indent=2)
print("\n💾 Saved: results/foundation_only_metrics.json")

In [ ]:
os.makedirs(f"{REPO_ROOT}/results", exist_ok=True)

day3_metadata = {
    "foundation_model": "dinov2_vits14",
    "embedding_dim": embedding_dim,
    "frozen": True,
    "device_used": "cuda (Tesla T4)",
    "cache_location": EMB_DIR,
    "cache_files": ["train_embeddings.pt", "val_embeddings.pt", "test_embeddings.pt"],
    "counts": {"train": len(train_cache["filepath"]), "val": len(val_cache["filepath"]), "test": len(test_cache["filepath"])},
    "integrity_check": {
        "overall": "PASS" if overall_pass else "FAIL",
        "train": train_issues, "val": val_issues, "test": test_issues,
    },
}

with open(f"{REPO_ROOT}/results/day3_foundation_embedding_metadata.json", "w") as f:
    json.dump(day3_metadata, f, indent=2)

print("✅ Metadata saved:", f"{REPO_ROOT}/results/day3_foundation_embedding_metadata.json")

In [ ]:
%cd {REPO_ROOT}
!git add results/day3_foundation_embedding_metadata.json
!git commit -m "Day 3: DINOv2 foundation embedding extraction and cache validation"
!git push

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, yaml, json, pandas as pd, numpy as np, torch
import torch.nn as nn
import torchvision.models as models
import torchvision.transforms as T
from PIL import Image

REPO_ROOT = "/content/pathology_project"
if not os.path.exists(REPO_ROOT):
    !git clone https://github.com/abdul-rehman-24/breast-histopathology-patch-classification.git {REPO_ROOT}
else:
    %cd {REPO_ROOT}
    !git pull

train_df = pd.read_csv(f"{REPO_ROOT}/splits/train.csv")
val_df   = pd.read_csv(f"{REPO_ROOT}/splits/val.csv")
test_df  = pd.read_csv(f"{REPO_ROOT}/splits/test.csv")
print(f"✅ Splits loaded -> train={len(train_df)}, val={len(val_df)}, test={len(test_df)}")

!pip install -q kagglehub
import kagglehub
DATA_PATH = kagglehub.dataset_download("ambarish/breakhis")
sample_path = train_df.iloc[0]["filepath"]
DATA_ROOT_FIX = None
if not os.path.exists(sample_path):
    old_root = sample_path.split("BreaKHis_v1")[0].rstrip("/")
    DATA_ROOT_FIX = (old_root, DATA_PATH)
    print(f"⚠️ Remap: {old_root} -> {DATA_PATH}")
else:
    print("✅ Paths valid")

device = "cuda" if torch.cuda.is_available() else "cpu"

# ---- LOAD Day-2 CNN checkpoint ----
CNN_CKPT_PATH = "/content/drive/MyDrive/pathology_project/checkpoints/cnn_baseline/best.pt"
def build_efficientnet_b0(num_classes=2):
    model = models.efficientnet_b0(weights=None)
    in_features = model.classifier[1].in_features
    model.classifier[1] = nn.Linear(in_features, num_classes)
    return model

cnn_model = build_efficientnet_b0().to(device)
ckpt = torch.load(CNN_CKPT_PATH, map_location=device, weights_only=False)
cnn_model.load_state_dict(ckpt["model_state"])
cnn_model.eval()
print(f"✅ Day-2 CNN checkpoint loaded (epoch {ckpt['epoch']+1}, val_auroc={ckpt['val_auroc']:.4f})")

# ---- LOAD Day-3 DINOv2 embedding cache ----
EMB_DIR = "/content/drive/MyDrive/pathology_project/embeddings/dinov2_vits14"
train_emb_cache = torch.load(f"{EMB_DIR}/train_embeddings.pt", weights_only=False)
val_emb_cache   = torch.load(f"{EMB_DIR}/val_embeddings.pt", weights_only=False)
test_emb_cache  = torch.load(f"{EMB_DIR}/test_embeddings.pt", weights_only=False)
print(f"✅ Foundation embeddings loaded -> train={len(train_emb_cache['filepath'])}, val={len(val_emb_cache['filepath'])}, test={len(test_emb_cache['filepath'])}")

In [ ]:
# ---- Modify CNN to expose features (before classifier) ----
class CNNFeatureExtractor(nn.Module):
    def __init__(self, trained_model):
        super().__init__()
        self.features = trained_model.features
        self.avgpool = trained_model.avgpool
    def forward(self, x):
        x = self.features(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        return x

cnn_feat_extractor = CNNFeatureExtractor(cnn_model).to(device).eval()

# ---- Test on ONE sample ----
sample_path = train_df.iloc[0]["filepath"]
real_path = sample_path.replace(DATA_ROOT_FIX[0], DATA_ROOT_FIX[1]) if DATA_ROOT_FIX else sample_path
img = Image.open(real_path).convert("RGB")

cnn_transform = T.Compose([
    T.Resize((224, 224)), T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])
img_tensor = cnn_transform(img).unsqueeze(0).to(device)

with torch.no_grad():
    cnn_feat = cnn_feat_extractor(img_tensor)

# ---- Get corresponding foundation embedding from cache ----
idx_in_cache = train_emb_cache["filepath"].index(sample_path)
foundation_feat = train_emb_cache["embedding"][idx_in_cache]

print(f"✅ Image: {sample_path}")
print(f"✅ Label: {train_df.iloc[0]['label']}, Group: {train_df.iloc[0]['group_id']}")
print(f"✅ Cached label: {train_emb_cache['label'][idx_in_cache]}, Cached group: {train_emb_cache['group_id'][idx_in_cache]}")
print(f"✅ CNN feature shape: {cnn_feat.shape}")
print(f"✅ Foundation feature shape: {foundation_feat.shape}")
print(f"✅ Same image confirmed (label+group match): {train_df.iloc[0]['label']==train_emb_cache['label'][idx_in_cache] and train_df.iloc[0]['group_id']==train_emb_cache['group_id'][idx_in_cache]}")

In [ ]:
from torch.utils.data import Dataset, DataLoader

CNN_FEAT_DIR = "/content/drive/MyDrive/pathology_project/embeddings/cnn_efficientnet_b0"
os.makedirs(CNN_FEAT_DIR, exist_ok=True)

class ExtractionDataset(Dataset):
    def __init__(self, df, transform):
        self.df = df.reset_index(drop=True)
        self.transform = transform
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        path = row["filepath"]
        real_path = path.replace(DATA_ROOT_FIX[0], DATA_ROOT_FIX[1]) if DATA_ROOT_FIX else path
        img = Image.open(real_path).convert("RGB")
        return self.transform(img), path, row["group_id"], row["label"]

def extract_cnn_features(df, split_name, batch_size=64, save_every=10):
    cache_path = f"{CNN_FEAT_DIR}/{split_name}_cnn_features.pt"
    if os.path.exists(cache_path):
        cache = torch.load(cache_path, weights_only=False)
        done_paths = set(cache["filepath"])
    else:
        cache = {"filepath": [], "group_id": [], "label": [], "feature": []}
        done_paths = set()

    remaining_df = df[~df["filepath"].isin(done_paths)].reset_index(drop=True)
    print(f"📊 {split_name}: {len(done_paths)}/{len(df)} cached, {len(remaining_df)} remaining")
    if len(remaining_df) == 0:
        print(f"✅ {split_name}: already complete")
        return cache

    ds = ExtractionDataset(remaining_df, cnn_transform)
    loader = DataLoader(ds, batch_size=batch_size, shuffle=False, num_workers=2)
    batch_count = 0
    for imgs, paths, gids, labels in loader:
        imgs = imgs.to(device)
        with torch.no_grad():
            feats = cnn_feat_extractor(imgs).cpu()
        cache["filepath"].extend(list(paths))
        cache["group_id"].extend(list(gids))
        cache["label"].extend(list(labels))
        cache["feature"].extend(list(feats))
        batch_count += 1
        if batch_count % save_every == 0:
            torch.save(cache, cache_path)
            print(f"  💾 {len(cache['filepath'])}/{len(df)} processed")
    torch.save(cache, cache_path)
    print(f"✅ {split_name} COMPLETE: {len(cache['filepath'])}/{len(df)} -> {cache_path}")
    return cache

train_cnn_cache = extract_cnn_features(train_df, "train")
val_cnn_cache   = extract_cnn_features(val_df, "val")
test_cnn_cache  = extract_cnn_features(test_df, "test")

In [ ]:
def check_alignment(df, cnn_cache, emb_cache, split_name):
    issues = []
    cnn_df = pd.DataFrame({"filepath": cnn_cache["filepath"], "cnn_label": cnn_cache["label"], "cnn_group": cnn_cache["group_id"]})
    emb_df = pd.DataFrame({"filepath": emb_cache["filepath"], "emb_label": emb_cache["label"], "emb_group": emb_cache["group_id"]})

    merged = df[["filepath","label","group_id"]].merge(cnn_df, on="filepath", how="left").merge(emb_df, on="filepath", how="left")

    if merged["cnn_label"].isna().sum() > 0: issues.append(f"{merged['cnn_label'].isna().sum()} missing CNN features")
    if merged["emb_label"].isna().sum() > 0: issues.append(f"{merged['emb_label'].isna().sum()} missing Foundation features")
    if (merged["label"] != merged["cnn_label"]).sum() > 0: issues.append("CNN label mismatch")
    if (merged["label"] != merged["emb_label"]).sum() > 0: issues.append("Foundation label mismatch")
    if (merged["group_id"] != merged["cnn_group"]).sum() > 0: issues.append("CNN group mismatch")
    if (merged["group_id"] != merged["emb_group"]).sum() > 0: issues.append("Foundation group mismatch")
    if merged["filepath"].duplicated().sum() > 0: issues.append("duplicate filepaths")

    status = "PASS" if not issues else "FAIL"
    print(f"{'✅' if status=='PASS' else '❌'} {split_name} alignment: {status}")
    for i in issues: print(f"   ⚠️ {i}")
    return status == "PASS"

train_align = check_alignment(train_df, train_cnn_cache, train_emb_cache, "train")
val_align   = check_alignment(val_df, val_cnn_cache, val_emb_cache, "val")
test_align  = check_alignment(test_df, test_cnn_cache, test_emb_cache, "test")

print(f"\n🧪 OVERALL ALIGNMENT: {'✅ PASS' if (train_align and val_align and test_align) else '❌ FAIL'}")

In [ ]:
class ProjectionHead(nn.Module):
    def __init__(self, in_dim, out_dim=256):
        super().__init__()
        self.proj = nn.Sequential(
            nn.Linear(in_dim, out_dim),
            nn.ReLU(),
            nn.Dropout(0.3)
        )
    def forward(self, x):
        return self.proj(x)

cnn_projection = ProjectionHead(in_dim=1280, out_dim=256).to(device)
foundation_projection = ProjectionHead(in_dim=384, out_dim=256).to(device)

# ---- Test on sample ----
with torch.no_grad():
    proj_cnn = cnn_projection(cnn_feat)
    proj_found = foundation_projection(foundation_feat.unsqueeze(0).to(device))

print(f"✅ Projected CNN shape: {proj_cnn.shape}")
print(f"✅ Projected Foundation shape: {proj_found.shape}")

In [ ]:
fused = torch.cat([proj_cnn, proj_found], dim=1)
print(f"✅ Fused feature shape: {fused.shape}")  # expect (1, 512)

In [ ]:
class FusionModel(nn.Module):
    def __init__(self, cnn_dim=1280, foundation_dim=384, proj_dim=256, num_classes=2):
        super().__init__()
        self.cnn_projection = ProjectionHead(cnn_dim, proj_dim)
        self.foundation_projection = ProjectionHead(foundation_dim, proj_dim)
        self.classifier = nn.Sequential(
            nn.Linear(proj_dim * 2, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes)
        )
    def forward(self, cnn_feat, foundation_feat):
        proj_cnn = self.cnn_projection(cnn_feat)
        proj_found = self.foundation_projection(foundation_feat)
        fused = torch.cat([proj_cnn, proj_found], dim=1)
        out = self.classifier(fused)
        return out

fusion_model = FusionModel().to(device)

# ---- Sanity check forward pass ----
with torch.no_grad():
    out = fusion_model(cnn_feat, foundation_feat.unsqueeze(0).to(device))
print(f"✅ Fusion model output shape: {out.shape}")  # expect (1, 2)
print(f"✅ Total fusion model params: {sum(p.numel() for p in fusion_model.parameters()):,}")

In [ ]:
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import roc_auc_score
import torch.optim as optim
import time

LABEL_MAP = {"benign": 0, "malignant": 1}

class FusedFeatureDataset(Dataset):
    def __init__(self, cnn_cache, emb_cache):
        # align by filepath -> build lookup for embedding cache
        emb_lookup = {p: i for i, p in enumerate(emb_cache["filepath"])}
        self.cnn_features = cnn_cache["feature"]
        self.foundation_features = []
        self.labels = []
        for i, path in enumerate(cnn_cache["filepath"]):
            j = emb_lookup[path]
            self.foundation_features.append(emb_cache["embedding"][j])
            self.labels.append(LABEL_MAP[cnn_cache["label"][i]])

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.cnn_features[idx], self.foundation_features[idx], self.labels[idx]

train_fused_ds = FusedFeatureDataset(train_cnn_cache, train_emb_cache)
val_fused_ds   = FusedFeatureDataset(val_cnn_cache, val_emb_cache)
test_fused_ds  = FusedFeatureDataset(test_cnn_cache, test_emb_cache)

train_fused_loader = DataLoader(train_fused_ds, batch_size=64, shuffle=True)
val_fused_loader   = DataLoader(val_fused_ds, batch_size=64, shuffle=False)

print(f"✅ Fused datasets ready -> train={len(train_fused_ds)}, val={len(val_fused_ds)}, test={len(test_fused_ds)}")

# ---- Training config ----
SEED = 42
torch.manual_seed(SEED)
EPOCHS = 20  # small model, cached features -> can afford more epochs, cheap
LR = 1e-3

FUSION_CKPT_DIR = "/content/drive/MyDrive/pathology_project/checkpoints/fusion_model"
os.makedirs(FUSION_CKPT_DIR, exist_ok=True)

class_counts = pd.Series(train_cnn_cache["label"]).value_counts()
w_benign = len(train_cnn_cache["label"]) / (2 * class_counts["benign"])
w_malignant = len(train_cnn_cache["label"]) / (2 * class_counts["malignant"])
class_weights = torch.tensor([w_benign, w_malignant], dtype=torch.float32).to(device)

criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = optim.AdamW(fusion_model.parameters(), lr=LR)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=3, factor=0.5)

fusion_train_config = {
    "seed": SEED, "epochs": EPOCHS, "lr": LR, "batch_size": 64,
    "proj_dim": 256, "optimizer": "AdamW", "loss": "CrossEntropyLoss (weighted)",
    "cnn_frozen": True, "foundation_frozen": True, "selection_metric": "val_auroc",
}

# ---- Resume support ----
start_epoch = 0
best_val_auroc = 0.0
latest_path = f"{FUSION_CKPT_DIR}/latest.pt"
if os.path.exists(latest_path):
    ckpt = torch.load(latest_path, map_location=device, weights_only=False)
    fusion_model.load_state_dict(ckpt["model_state"])
    optimizer.load_state_dict(ckpt["optimizer_state"])
    scheduler.load_state_dict(ckpt["scheduler_state"])
    start_epoch = ckpt["epoch"] + 1
    best_val_auroc = ckpt["best_val_auroc"]
    print(f"🔄 Resuming from epoch {start_epoch}")
else:
    print("🆕 Starting fresh")

def run_fusion_epoch(loader, train_mode=True):
    fusion_model.train() if train_mode else fusion_model.eval()
    total_loss = 0
    all_labels, all_probs = [], []
    with torch.set_grad_enabled(train_mode):
        for cnn_f, found_f, labels in loader:
            cnn_f, found_f, labels = cnn_f.to(device), found_f.to(device), labels.to(device)
            if train_mode: optimizer.zero_grad()
            out = fusion_model(cnn_f, found_f)
            loss = criterion(out, labels)
            if train_mode:
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * cnn_f.size(0)
            probs = torch.softmax(out, dim=1)[:, 1]
            all_labels.extend(labels.cpu().numpy())
            all_probs.extend(probs.detach().cpu().numpy())
    avg_loss = total_loss / len(loader.dataset)
    auroc = roc_auc_score(all_labels, all_probs)
    return avg_loss, auroc

for epoch in range(start_epoch, EPOCHS):
    t0 = time.time()
    train_loss, train_auroc = run_fusion_epoch(train_fused_loader, True)
    val_loss, val_auroc = run_fusion_epoch(val_fused_loader, False)
    scheduler.step(val_loss)

    print(f"Epoch {epoch+1}/{EPOCHS} | train_loss={train_loss:.4f} train_auroc={train_auroc:.4f} "
          f"| val_loss={val_loss:.4f} val_auroc={val_auroc:.4f} | {time.time()-t0:.1f}s")

    torch.save({
        "epoch": epoch, "model_state": fusion_model.state_dict(),
        "optimizer_state": optimizer.state_dict(), "scheduler_state": scheduler.state_dict(),
        "val_auroc": float(val_auroc), "best_val_auroc": float(max(best_val_auroc, val_auroc)),
    }, latest_path)

    if val_auroc > best_val_auroc:
        best_val_auroc = val_auroc
        torch.save({"epoch": epoch, "model_state": fusion_model.state_dict(), "val_auroc": float(val_auroc)},
                   f"{FUSION_CKPT_DIR}/best.pt")
        print(f"  💾 New best (val_auroc={val_auroc:.4f})")

with open(f"{REPO_ROOT}/configs/fusion_training_config.json", "w") as f:
    json.dump(fusion_train_config, f, indent=2)

print(f"\n✅ Training complete. Best val_auroc: {best_val_auroc:.4f}")

In [ ]:
import numpy as np
from sklearn.metrics import (roc_auc_score, average_precision_score, f1_score,
                               precision_score, accuracy_score, balanced_accuracy_score, confusion_matrix)
import matplotlib.pyplot as plt

best_ckpt = torch.load(f"{FUSION_CKPT_DIR}/best.pt", map_location=device, weights_only=False)
fusion_model.load_state_dict(best_ckpt["model_state"])
fusion_model.eval()
print(f"✅ Loaded best fusion checkpoint: epoch {best_ckpt['epoch']+1}, val_auroc={best_ckpt['val_auroc']:.4f}")

test_fused_loader = DataLoader(test_fused_ds, batch_size=64, shuffle=False)
all_labels, all_probs, all_preds = [], [], []
with torch.no_grad():
    for cnn_f, found_f, labels in test_fused_loader:
        cnn_f, found_f = cnn_f.to(device), found_f.to(device)
        out = fusion_model(cnn_f, found_f)
        probs = torch.softmax(out, dim=1)[:, 1]
        preds = (probs > 0.5).long()
        all_labels.extend(labels.numpy())
        all_probs.extend(probs.cpu().numpy())
        all_preds.extend(preds.cpu().numpy())

all_labels, all_probs, all_preds = np.array(all_labels), np.array(all_probs), np.array(all_preds)
cm = confusion_matrix(all_labels, all_preds)
tn, fp, fn, tp = cm.ravel()

fusion_metrics = {
    "auroc": roc_auc_score(all_labels, all_probs),
    "auprc": average_precision_score(all_labels, all_probs),
    "f1": f1_score(all_labels, all_preds),
    "precision": precision_score(all_labels, all_preds),
    "sensitivity_recall": tp/(tp+fn),
    "specificity": tn/(tn+fp),
    "accuracy": accuracy_score(all_labels, all_preds),
    "balanced_accuracy": balanced_accuracy_score(all_labels, all_preds),
    "confusion_matrix": cm.tolist(),
    "n_test_samples": len(all_labels),
    "checkpoint_epoch": best_ckpt["epoch"]+1,
    "checkpoint_val_auroc": float(best_ckpt["val_auroc"]),
}

print("\n📊 TEST METRICS (CNN+Foundation Fusion):")
for k, v in fusion_metrics.items():
    if k != "confusion_matrix":
        print(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")

with open(f"{REPO_ROOT}/results/fusion_metrics.json", "w") as f:
    json.dump(fusion_metrics, f, indent=2)

fig, ax = plt.subplots(figsize=(5,4))
im = ax.imshow(cm, cmap="Greens")
ax.set_xticks([0,1]); ax.set_yticks([0,1])
ax.set_xticklabels(["Benign","Malignant"]); ax.set_yticklabels(["Benign","Malignant"])
ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")
ax.set_title("Fusion Model - Confusion Matrix (Test)")
for i in range(2):
    for j in range(2):
        ax.text(j, i, cm[i,j], ha="center", va="center")
plt.colorbar(im)
plt.savefig(f"{REPO_ROOT}/figures/fusion_confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()

print("\n💾 Saved: results/fusion_metrics.json, figures/fusion_confusion_matrix.png")

In [ ]:
with open(f"{REPO_ROOT}/results/cnn_baseline_metrics.json") as f:
    cnn_metrics = json.load(f)

comparison = pd.DataFrame({
    "CNN Baseline": {k: cnn_metrics[k] for k in ["auroc","auprc","f1","accuracy","balanced_accuracy","sensitivity_recall","specificity"]},
    "CNN+Foundation Fusion": {k: fusion_metrics[k] for k in ["auroc","auprc","f1","accuracy","balanced_accuracy","sensitivity_recall","specificity"]},
})
comparison["Delta (Fusion - CNN)"] = comparison["CNN+Foundation Fusion"] - comparison["CNN Baseline"]
print(comparison.round(4))

with open(f"{REPO_ROOT}/results/day4_comparison.json", "w") as f:
    json.dump(comparison.round(4).to_dict(), f, indent=2)

In [ ]:
comparison_3way = pd.DataFrame({
    "CNN Baseline": {k: cnn_metrics[k] for k in ["auroc","auprc","f1","accuracy","balanced_accuracy","sensitivity_recall","specificity"]},
    "Foundation-Only": {k: foundation_only_metrics[k] for k in ["auroc","auprc","f1","accuracy","balanced_accuracy","sensitivity_recall","specificity"]},
    "CNN+Foundation Fusion": {k: fusion_metrics[k] for k in ["auroc","auprc","f1","accuracy","balanced_accuracy","sensitivity_recall","specificity"]},
})
print(comparison_3way.round(4))

with open(f"{REPO_ROOT}/results/day4_comparison_3way.json", "w") as f:
    json.dump(comparison_3way.round(4).to_dict(), f, indent=2)
print("\n💾 Saved: results/day4_comparison_3way.json")

In [ ]:
fusion_experiment_record = {
    "experiment_name": "cnn_foundation_fusion",
    "cnn_model": "EfficientNet-B0 (frozen, Day-2 checkpoint)",
    "foundation_model": "DINOv2 ViT-S/14 (frozen, Day-3 cache)",
    "cnn_feature_dim": 1280, "foundation_feature_dim": 384, "projection_dim": 256, "fused_dim": 512,
    "fusion_method": "concatenation + shallow classifier (Linear-ReLU-Dropout-Linear)",
    "classifier_trainable_params": 492418,
    "dataset_split_version": "day1_patient_wise_70_15_15_seed42",
    "seed": 42,
    "training_config": fusion_train_config,
    "checkpoint_used": f"best.pt (epoch {best_ckpt['epoch']+1})",
    "checkpoint_location": f"{FUSION_CKPT_DIR}/best.pt (Drive)",
    "validation_results": {"val_auroc": float(best_ckpt["val_auroc"])},
    "test_results": fusion_metrics,
    "comparison_3way": comparison_3way.round(4).to_dict(),
    "notes": "Fusion model overfit heavily after epoch ~5 (train_auroc->1.0). Best selected via val AUROC only. "
             "Foundation-only baseline also evaluated (test AUROC=0.9054, F1=0.9011). "
             "3-way comparison: Fusion achieves best AUROC/F1/accuracy/sensitivity but slightly lower "
             "specificity than CNN alone. Foundation-only model showed strong validation AUROC (0.9887) "
             "but similar overfitting pattern to CNN/Fusion when generalizing to test."
}
with open(f"{REPO_ROOT}/results/fusion_experiment_record.json", "w") as f:
    json.dump(fusion_experiment_record, f, indent=2)
print("✅ fusion_experiment_record created and saved")

foundation_only_record = {
    "experiment_name": "foundation_only_baseline",
    "foundation_model": "DINOv2 ViT-S/14 (frozen)",
    "projection_dim": 256,
    "seed": 42, "epochs": 20, "lr": 1e-3,
    "checkpoint_used": f"best.pt (epoch {best_fo_ckpt['epoch']+1})",
    "checkpoint_location": f"{FO_CKPT_DIR}/best.pt (Drive)",
    "validation_results": {"val_auroc": float(best_fo_ckpt["val_auroc"])},
    "test_results": foundation_only_metrics,
}
with open(f"{REPO_ROOT}/results/foundation_only_experiment_record.json", "w") as f:
    json.dump(foundation_only_record, f, indent=2)
print("✅ foundation_only_record created and saved")

%cd {REPO_ROOT}
!git add configs/fusion_training_config.json results/fusion_metrics.json results/day4_comparison_3way.json results/fusion_experiment_record.json results/foundation_only_metrics.json results/foundation_only_experiment_record.json figures/fusion_confusion_matrix.png
!git commit -m "Day 4: CNN-foundation fusion + foundation-only baseline + 3-way comparison"
!git push

In [ ]:
import os
os.makedirs(f"{REPO_ROOT}/notebooks", exist_ok=True)

from google.colab import _message
notebook_json = _message.blocking_request('get_ipynb', timeout_sec=30)

import json
with open(f"{REPO_ROOT}/notebooks/day1_to_day4.ipynb", "w") as f:
    json.dump(notebook_json['ipynb'], f)

print("✅ Notebook saved to repo")